# 03 · Model training & tuning

The full run is `python -m src.training.train`. This notebook loads its persisted outputs rather than retraining.

* Hyper-parameter search: Grid (LR, DT, SVM, KNN) or Randomized (RF, HistGB, XGBoost), **StratifiedKFold(3)** on a 50k stratified subsample of the training split, scored by **average precision (PR-AUC)**, not accuracy.
* Class imbalance: `class_weight='balanced'` / `scale_pos_weight`.
* Final refit on the full training split (KNN capped at 40k rows for inference cost; SVM is a calibrated linear SVM because kernel SVMs do not scale to 150k rows).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option('display.max_columns', 40)

In [2]:
from src.config import MODELS_DIR
reg = json.loads((MODELS_DIR/'registry.json').read_text())
m = json.loads((MODELS_DIR/reg['diabetes_risk']['path']/'metrics.json').read_text())
print(m['tuning'])

{'rows': 50000, 'cv': 'StratifiedKFold(3, shuffle)', 'scoring': 'average_precision', 'quick_mode': False}


In [3]:
pd.DataFrame({k: {'best_params': v['best_params'], 'cv_pr_auc': f"{v['cv_pr_auc_mean']:.4f} ± {v['cv_pr_auc_std']:.4f}", 'train_rows': v['train_rows'], 'seconds': v['train_seconds']} for k, v in m['models'].items()}).T

,best_params,cv_pr_auc,train_rows,seconds
logistic_regression,{'C': 10.0},0.4233 ± 0.0050,152208,16.1
decision_tree,"{'max_depth': 8, 'min_samples_leaf': 400}",0.4011 ± 0.0045,152208,9.8
random_forest,"{'min_samples_leaf': 20, 'max_features': 'sqrt...",0.4321 ± 0.0071,152208,117.1
svm_linear,{'C': 1.0},0.4240 ± 0.0054,152208,3.7
knn,"{'n_neighbors': 151, 'weights': 'uniform'}",0.4169 ± 0.0083,40000,28.4
gradient_boosting,"{'l2_regularization': 5.872283616443723, 'lear...",0.4364 ± 0.0088,152208,22.5
xgboost,"{'colsample_bytree': 0.6163856702151521, 'lear...",0.4393 ± 0.0093,152208,207.5


Model selection uses **validation PR-AUC**. The test split is never used to choose a model or a threshold.

In [4]:
pd.Series({k: v['validation']['pr_auc'] for k, v in m['models'].items()}).sort_values(ascending=False)

xgboost                0.4362
gradient_boosting      0.4334
random_forest          0.4261
svm_linear             0.4138
logistic_regression    0.4133
knn                    0.4105
decision_tree          0.4016
dtype: float64

In [5]:
print('Selected:', m['selected_model'])

Selected: xgboost
